# Chapter 3 Practical 05: Evaluation with MAE, Precision@K, and Recall@K

Learning objectives:
- Create a leave-one-out test split.
- Evaluate rating prediction with MAE.
- Evaluate Top-K recommendation with Precision@K, Recall@K, and HitRate@K.
- Compare user-user and item-item CF against a popularity baseline.

Slide connection: practical exercise evaluation requirements and performance comparison.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "ratings_chapter3.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "ratings_chapter3.csv").exists():
    DATA_DIR = Path("chapter_03_collaborative_filtering/data")

ratings = pd.read_csv(DATA_DIR / "ratings_chapter3.csv")
movies = pd.read_csv(DATA_DIR / "movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


In [ ]:
np.random.seed(7)
test_rows = ratings_named.groupby("user_id", group_keys=False).sample(n=1, random_state=7)
train_rows = ratings_named.drop(test_rows.index)
train_matrix = train_rows.pivot_table(index="user_id", columns="title", values="rating")

test_rows[["user_id", "title", "rating"]].sort_values("user_id")


In [ ]:
def pearson_on_overlap(matrix, user_a, user_b):
    if user_a not in matrix.index or user_b not in matrix.index:
        return np.nan
    pair = matrix.loc[[user_a, user_b]].dropna(axis=1)
    if pair.shape[1] < 2:
        return np.nan
    if pair.loc[user_a].std() == 0 or pair.loc[user_b].std() == 0:
        return np.nan
    return float(np.corrcoef(pair.loc[user_a], pair.loc[user_b])[0, 1])

def predict_user_user_eval(matrix, target_user, item, k=3):
    if target_user not in matrix.index or item not in matrix.columns:
        return np.nan
    rows = []
    for other in matrix.index.drop(target_user):
        if pd.isna(matrix.loc[other, item]):
            continue
        sim = pearson_on_overlap(matrix, target_user, other)
        if not pd.isna(sim) and sim > 0:
            rows.append((other, sim))
    rows = sorted(rows, key=lambda x: x[1], reverse=True)[:k]
    if not rows:
        return np.nan
    target_mean = matrix.loc[target_user].mean()
    numerator = sum(sim * (matrix.loc[u, item] - matrix.loc[u].mean()) for u, sim in rows)
    denominator = sum(abs(sim) for _, sim in rows)
    return target_mean + numerator / denominator if denominator else np.nan

def build_item_sim(matrix):
    sim = pd.DataFrame(index=matrix.columns, columns=matrix.columns, dtype=float)
    for a in matrix.columns:
        for b in matrix.columns:
            pair = matrix[[a, b]].dropna()
            sim.loc[a, b] = 1.0 if a == b else (
                np.corrcoef(pair[a], pair[b])[0, 1]
                if len(pair) >= 2 and pair[a].std() != 0 and pair[b].std() != 0
                else np.nan
            )
    return sim

train_item_sim = build_item_sim(train_matrix)

def predict_item_item_eval(matrix, user, item, k=3):
    if user not in matrix.index or item not in train_item_sim.index:
        return np.nan
    rated = matrix.loc[user].dropna()
    rows = []
    for rated_item, rating in rated.items():
        sim = train_item_sim.loc[item, rated_item]
        if not pd.isna(sim) and sim > 0:
            rows.append((rated_item, rating, sim))
    rows = sorted(rows, key=lambda x: x[2], reverse=True)[:k]
    if not rows:
        return np.nan
    numerator = sum(rating * sim for _, rating, sim in rows)
    denominator = sum(abs(sim) for _, _, sim in rows)
    return numerator / denominator if denominator else np.nan


In [ ]:
def popularity_prediction(matrix, item):
    return matrix[item].mean() if item in matrix.columns else matrix.stack().mean()

pred_rows = []
for _, row in test_rows.iterrows():
    user, item, actual = row["user_id"], row["title"], row["rating"]
    for model, pred in [
        ("user_user", predict_user_user_eval(train_matrix, user, item)),
        ("item_item", predict_item_item_eval(train_matrix, user, item)),
        ("popularity", popularity_prediction(train_matrix, item)),
    ]:
        if not pd.isna(pred):
            pred_rows.append({"model": model, "user": user, "item": item, "actual": actual, "predicted": pred})

predictions = pd.DataFrame(pred_rows)
predictions["absolute_error"] = (predictions["actual"] - predictions["predicted"]).abs()
predictions.groupby("model")["absolute_error"].mean().rename("MAE").round(3)


In [ ]:
def topk_popularity(matrix, user, k=3):
    seen = set(matrix.loc[user].dropna().index)
    scores = matrix.mean().drop(labels=list(seen), errors="ignore")
    return scores.sort_values(ascending=False).head(k).index.tolist()

def topk_user_user(matrix, user, k=3):
    unseen = matrix.columns[matrix.loc[user].isna()]
    rows = [(item, predict_user_user_eval(matrix, user, item)) for item in unseen]
    rows = [(item, score) for item, score in rows if not pd.isna(score)]
    return [item for item, _ in sorted(rows, key=lambda x: x[1], reverse=True)[:k]]

def evaluate_topk(model_fn, train_matrix, test_rows, k=3, relevant_threshold=5):
    rows = []
    for user, user_test in test_rows.groupby("user_id"):
        relevant = set(user_test.loc[user_test["rating"] >= relevant_threshold, "title"])
        recommended = model_fn(train_matrix, user, k)
        hits = len(set(recommended) & relevant)
        rows.append({
            "user": user,
            "precision_at_k": hits / k,
            "recall_at_k": hits / len(relevant) if relevant else np.nan,
            "hit_rate_at_k": 1 if hits > 0 else 0,
            "recommended": recommended,
            "relevant": sorted(relevant),
        })
    return pd.DataFrame(rows)

pop_eval = evaluate_topk(topk_popularity, train_matrix, test_rows)
uu_eval = evaluate_topk(topk_user_user, train_matrix, test_rows)

pd.DataFrame({
    "model": ["popularity", "user_user"],
    "precision_at_3": [pop_eval["precision_at_k"].mean(), uu_eval["precision_at_k"].mean()],
    "recall_at_3": [pop_eval["recall_at_k"].mean(), uu_eval["recall_at_k"].mean()],
    "hit_rate_at_3": [pop_eval["hit_rate_at_k"].mean(), uu_eval["hit_rate_at_k"].mean()],
}).round(3)


Exercises:
1. Change the relevant threshold from 5 to 6.
2. Use a larger test split and compare whether the model ranking changes.
